# 第6回 分類の判定を、混同行列で評価する（1行ずつ読む版）（解答例・教員用）


---


## 前回からの接続


第5回は、肉に当てた光のスペクトルから、脂肪の割合を当てる式を作りました。当てたいものは、脂肪の割合のような「量」でした。

今日は、当てたいものが「迷惑メッセージか、そうでないか」の2択です。2択を当てることを、分類（classification）と呼びます。分類では、当たったか外れたかを、どう評価すればよいでしょうか。


---


## 今日の分析目標


本文から、迷惑メッセージをどう見分け、その見分け方をどう評価するかに答える。

携帯電話に届いたSMSの本文から、迷惑メッセージかどうかを判定します。まず、簡単なルールで判定し、正解率で評価します。ところが、正解率だけでは、良いフィルターかどうかが分かりません。そこで、当たりと外れを4つのマスに分けて数え（混同行列）、見逃しと誤検出を分けて評価します。最後に、手がかりをまとめて迷惑である確率を出し（ロジスティック回帰）、どこで切るかを動かします。

もう1つのねらいは、条件で処理を分ける書き方（if）を使い、ルールや判定のコードを読めるようになることです。「freeを含むなら迷惑」「数字が5字以上なら迷惑」を、ifで書きます。お手本ごとに、コードを1行ずつ読み（「1行ずつ読む」）、くり返しを少しだけ回して、変数の中身を見ます（「中身を見る」）。


### 公開データの世界

第5回は、食品の分析のデータで、数の列が100本並ぶ表でした。今日は、文章のデータです。数の表ではなく、人が書いたメッセージそのものです。データの形が変われば、道具の組み立て方も変わります。


### 場面

「当選しました。この番号に電話を」のような迷惑メッセージ（spam）が、携帯電話には毎日のように届きます。携帯電話の会社やアプリは、届く前に迷惑フォルダへ振り分けています。この仕組みを、迷惑メッセージのフィルターと呼びます。

届いたメッセージの本文から、迷惑か普通かを判定したい。答えは2つのうちのどちらかで、量ではなく2択を当てます（分類）。そして、作ったフィルターがどのくらい良いかも評価したい。

フィルターの誤りは2種類あります。迷惑メッセージを通してしまう誤り（見逃し）と、大事な連絡を迷惑フォルダに入れてしまう誤り（誤検出）です。どちらも困りますが、困り方は違います。


### 手元のデータ

SMS Spam Collectionは、研究のために集めた英語のSMS 5,574通のデータです。1行が1通です。

- `label`：迷惑（spam）か普通（ham）か
- `text`：本文

迷惑メッセージは、英国の利用者が迷惑SMSを投稿する掲示板などから集めたものです。普通のメッセージは、シンガポール国立大学の学生が、公開されることを承知して提供したものなどです。

データはUCI機械学習リポジトリで公開されていて（CC BY 4.0）、出典（Almeida, Gómez Hidalgo, Yamakami, 2011）を書けば使えます。


### 最後に出す答え

迷惑メッセージを、どれだけ捕まえ、普通のメッセージをどれだけ誤って捕まえるか。そして、そのフィルターを、どんな数で評価すればよいかです。


### 今日の一周（結果だけ先に）

コードの説明はあとにして、今日の解析を、結果だけで一度通して見ておきます。

1. ルールで見分ける：「freeを含むなら迷惑」と判定すると、正解率は0.894です。
2. 正解率を疑う：何も迷惑と言わないルールでも、正解率は0.874です。正解率だけでは分かりません。当たりと外れを4つのマスで数えると、freeのルールは、迷惑653通のうち484通を見逃しています。
3. 2つの割合：本文の数字を数え、5字以上なら迷惑とすると、見逃し96通、誤検出39通まで減ります。
4. 確率を出す：数字と15語から、ロジスティック回帰で迷惑である確率を出し、0.5で切ると、見逃し83通、誤検出23通です。数字のルールより、どちらも少なくなります。
5. しきい値：切る場所（しきい値）を動かすと、見逃しと誤検出が入れ替わります。全部のしきい値を1本の線（ROC曲線）にし、下の面積（AUC）で測ると0.981です。ところが、えっ、迷惑なのに、迷惑である確率が0.05より小さい通が24通あります。しきい値をどこに動かしても、捕まえられません。

このあと、この5つの段の中身を、1つずつ開けていきます。


### 今日の問いとやること

| 段 | 問い | やること |
|---|---|---|
| ルールで見分ける（1・2節） | ルールで見分けられる？ | 「freeを含むなら迷惑」をifで書き、正解率を測る |
| 正解率を疑う（3節） | 正解率は信じてよい？ | 何も迷惑と言わないルールと比べ、4つのマスを数える |
| 2つの割合（4節） | 見逃しと誤検出は？ | 再現率と適合率を出し、数字を数えるルールと比べる |
| 確率を出す（5節） | 手がかりをまとめると？ | 数字と15語でロジスティック回帰の確率を出し、0.5で切る |
| しきい値（6節） | しきい値を動かすと？ | 切る場所を動かし、ROC曲線とAUCで測る。見逃した迷惑を読む |


### 今日の進め方

段ごとに、まずスライドで、その段の結論までを通して聞きます。段の終わりにノートへ移り、その段の節のセルを上から実行して、スライドと同じ数が出るかを確かめます。そして、段ごとに1つあるTODOに答えます。

TODOは、実行の前に結果を予想して書き、1か所を替えて実行し、予想と比べる形です。予想が外れても、かまいません。

正解率・混同行列・交差検証（cross-validation）などの計算には、`sklearn`の道具を使います。道具ごとに、中で何をしているかを、小さな例で確かめてから使います。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 「語を含むなら迷惑」のような判定のルールを、ifを使った関数で書ける
- 少ない側を見つけたいとき、正解率だけでは評価できない理由を、何も見分けないルールと比べて説明できる
- 混同行列の4つのマス（当たり2つ・見逃し・誤検出）を読み、再現率と適合率を出せる
- ロジスティック回帰で迷惑である確率を出し、しきい値で切って判定できる
- しきい値を動かすと、見逃しと誤検出が入れ替わることを、表とROC曲線で確かめ、AUCで測れる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出す
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
pd.set_option('display.max_colwidth', 80)
print('セットアップ完了')


### お手本：表を読み込む

- 役割：メッセージの表を読み込み、通数を数えて、先頭5行を見る
- 入力：データの置き場所`DATA_DIR`と、ファイル名`sms_spam.csv`
- 出力：表`raw`（5,574通）


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
raw = pd.read_csv(f'{DATA_DIR}/sms_spam.csv')
print(len(raw), '通')
raw.head()


1通の本文は、短い英語の文章です。略語や打ち間違いも多くあります。


### お手本：表を整える

- 役割：同じ本文を1通にまとめ、本文の列の名前を日本語にし、迷惑なら1・普通なら0の列「迷惑」を作る
- 入力：表`raw`
- 出力：表`df`と、通数・迷惑の通数

`drop_duplicates('text')`は、同じ本文の行を、1行だけ残して消します。`(df['label'] == 'spam')`は、迷惑ならTrue、普通ならFalseの列で、`astype(int)`で1と0に直します。


In [ ]:
df = raw.drop_duplicates('text').reset_index(drop=True)
df = df.rename(columns={'text': '本文'})
df['迷惑'] = (df['label'] == 'spam').astype(int)
print(len(raw), '通 →', len(df), '通')
print(df['迷惑'].sum(), '通が迷惑')


#### 1行ずつ読む

- 1行目：`raw`から、同じ本文の行を1行だけ残して消す（`drop_duplicates`）
- 続けて`reset_index(drop=True)`で、行の番号を0から振り直し、`df`にしまう
- 番号を振り直すと、行の番号が0から抜けなく並ぶ
- 2行目：列の名前textを、本文に替える（`rename`）
- 3行目：labelがspamならTrue、違えばFalse。`astype(int)`で1と0に直し、列「迷惑」にしまう
- 4行目：`len(表)`は行の数。整える前と後の通数を並べて出す
- 5行目：1と0の列の合計は、1の数。つまり迷惑の通数


#### 中身を見る：列「迷惑」


In [ ]:
df[['label', '迷惑']].head(3)


spamの行だけが1、hamの行は0になった


同じ本文が何度も届いた分を1通にまとめ、5,171通にしました。列「迷惑」は、迷惑なら1、普通なら0で、これが答えの列になります。


---


## 1. どんなメッセージが届く？


ここから、段1「ルールで見分ける」（1・2節）です。問いは、簡単なルールで、迷惑メッセージを見分けられるか、です。

まず、迷惑がどのくらいの割合で届いているかを見ます。


### お手本：迷惑の割合

- 役割：迷惑（1）と普通（0）の通数と、迷惑の割合を出す
- 入力：表`df`の列「迷惑」
- 出力：通数と割合

1と0の列の平均は、1の割合になります。


In [ ]:
print(df['迷惑'].value_counts())
print(round(df['迷惑'].mean(), 3))


#### 1行ずつ読む

- 1行目：`value_counts()`は、値ごとに行の数を数える。0（普通）と1（迷惑）の通数
- 2行目：`mean()`は平均。1と0の列の平均は、1の割合
- `round(…, 3)`で、小数第3位までに丸める


5,171通のうち、迷惑は653通、普通は4,518通です。迷惑の割合は0.126で、8通に1通ほどです。少ない側を見つけたい、というこの偏りが、今日の話の鍵になります。


### お手本：迷惑の本文を読む

- 役割：迷惑の行だけを取り出し、本文の先頭5通を見る
- 入力：表`df`
- 出力：迷惑メッセージの本文5通


In [ ]:
df[df['迷惑'] == 1]['本文'].head()


#### 1行ずつ読む

- `df['迷惑'] == 1`：迷惑ならTrue、普通ならFalseの列
- `df[Trueの列]`：Trueの行だけを取り出した表
- `['本文']`で本文の列を取り出し、`head()`で先頭の5通を見る


free（無料）、win（当選）、claim（受け取る）、call（電話を）のような決まり文句があり、電話番号や短い番号が並びます。この特徴を、ルールにできないでしょうか。


---


## 2. ルールで見分けられる？


迷惑の本文には、freeがよく出てきます。そこで、本文にfreeを含むなら迷惑、含まないなら普通、と判定してみます。条件によって、答えを分けたいのです。


### 道具：文字列の中に語があるか

`語 in 文字列`は、文字列の中にその語があればTrue、なければFalseになります。`lower()`は、大文字を小文字にそろえます。FREEもFreeも、freeとして探せます。


In [ ]:
print('free' in 'Are you free now?')   # True
print('free' in 'Tired.')              # False
print('FREE entry'.lower())            # free entry


### 新しい書き方：if

条件によって、動かす行を分けるときは、`if`を使います。

- `if 条件:`の下の字下げした行は、条件がTrueのときだけ動く
- `else:`の下の行は、条件がFalseのときだけ動く

下の例では、`x`が5以上なので、上の`print`だけが動きます。`x = 7`を`x = 3`に替えると、下の`print`が動きます。


In [ ]:
x = 7
if x >= 5:
    print('5以上')     # 条件がTrueのときだけ動く
else:
    print('5未満')     # 条件がFalseのときだけ動く


### お手本：ルールの関数

- 役割：本文`text`に語`word`を含めば1（迷惑と判定）、含まなければ0（普通と判定）を返す
- 入力：本文`text`、探す語`word`
- 出力：1か0


In [ ]:
def rule(text, word):
    if word in text.lower():
        return 1          # 迷惑と判定
    else:
        return 0          # 普通と判定

print(rule('FREE entry into our weekly comp', 'free'))
print(rule('Tired. I have not slept well', 'free'))


#### 1行ずつ読む

- 1行目：関数`rule`を作る。入力は本文`text`と、探す語`word`の2つ
- `def`の下の字下げした行が、関数の中身。呼び出したときに動く
- 2行目：本文を小文字にそろえ（`text.lower()`）、その中に`word`があるかを調べる
- あればTrue、なければFalse。これが`if`の条件
- 3行目：条件がTrueのとき、1を返す（迷惑と判定）
- 4・5行目：条件がFalseのとき、0を返す（普通と判定）
- `return`は、関数の答えを返して、関数を終える
- 7行目：FREEを含む本文で呼ぶ。小文字にするとfreeを含むので1
- 8行目：freeを含まない本文で呼ぶ。0


#### 中身を見る：条件の中身


In [ ]:
text = 'Call FREEPHONE 0800 542 0578 now!'
print(text.lower())
print('free' in text.lower())
print(rule(text, 'free'))


小文字にすると、freephoneの中にfreeが入っている。だからTrueで、1


1通めはFREEを含むので1、2通めは含まないので0になりました。


### ifの分かれ道

下の図は、実際のメッセージ3通が、ifの分かれ道でどちらへ進むかを描いたものです。1通ずつ、条件で道が分かれます。「Are you free now?」のような普通のメッセージも、freeを含むので、迷惑の道へ進みます。


In [ ]:
# @title 図：ifの分かれ道（中身は読み飛ばしてよい）
examples = ['FREE entry into our £250 weekly comp ...',
            'Are you free now?can i call now?',
            "Tired. I haven't slept well the past few nights."]
for text in examples:
    if 'free' in text.lower():
        print('True  → return 1（迷惑と判定）', text)
    else:
        print('False → return 0（普通と判定）', text)


### お手本：全部の通に判定を付ける

- 役割：1通ずつ`rule`で判定し、リストにためて、列「判定」にする
- 入力：表`df`の本文
- 出力：迷惑と判定した通数

くり返しの結果をリストにためるのは、第4回と同じ書き方です。


In [ ]:
pred = []
for text in df['本文']:
    pred.append(rule(text, 'free'))
df['判定'] = pred
print(df['判定'].sum(), '通を迷惑と判定')


#### 1行ずつ読む

- 1行目：空のリスト`pred`を作る。ここに判定をためていく
- 2行目：列「本文」から1通ずつ取り出して`text`に入れ、字下げした行をくり返す
- 3行目：その1通を`rule`で判定し（1か0）、`pred`の後ろに足す（`append`）
- 5,171回くり返すと、`pred`には5,171個の1と0が並ぶ
- 4行目：リスト`pred`を、表の列「判定」にする
- 5行目：1と0の列の合計は、迷惑と判定した通数


#### 中身を見る：3通だけ回す


In [ ]:
pred = []
for text in df['本文'].head(3):
    pred.append(rule(text, 'free'))
    print(pred)


1回ごとに、後ろに1つ足される。3通めはFreeで始まるので1


### お手本：正解率

- 役割：判定が当たった通の割合（正解率）を出す
- 入力：列「迷惑」（本当の答え）と列「判定」
- 出力：正解率

$$
\text{正解率} = \frac{\text{判定が当たった通数}}{\text{全部の通数}}
$$

`sklearn`の`accuracy_score(本当, 判定)`は、正解率を返す道具です。


In [ ]:
from sklearn.metrics import accuracy_score
print(round(accuracy_score(df['迷惑'], df['判定']), 3))


#### 1行ずつ読む

- 1行目：`sklearn`の評価の道具の中から、`accuracy_score`を呼び出す
- 2行目：本当の答え（列「迷惑」）と判定（列「判定」）を渡すと、正解率が返る
- `round(…, 3)`で、小数第3位までに丸めて表示する


#### accuracy_scoreがしていること

1通ずつ、本当の答えと判定を見比べ、同じなら当たりとします。当たりの通数を、全部の通数で割ります。たとえば、表の5通（行の番号0・1・2・8・75）なら、次のようになります。

| 行 | 本当 | 判定 | 当たり？ |
|---:|---:|---:|:---:|
| 0 | 0 | 0 | ○ |
| 1 | 0 | 0 | ○ |
| 2 | 1 | 1 | ○ |
| 8 | 1 | 0 | × |
| 75 | 0 | 1 | × |

5通のうち当たりは3通で、正解率は3 ÷ 5 = 0.6です。これを5,171通で行ったのが、上の数です。


In [ ]:
df.loc[[0, 1, 2, 8, 75], ['迷惑', '判定']]


#### 中身を見る：5通で正解率


In [ ]:
y = [0, 0, 1, 1, 0]      # 本当（行0・1・2・8・75）
p = [0, 0, 1, 0, 1]      # 判定
print(accuracy_score(y, p))


上の表の5通の本当と判定を、リストで渡した。5通のうち3通が当たりで、0.6


「freeを含むなら迷惑」だけで、正解率は0.894です。1行のルールで、9割ほどのメッセージを正しく判定できました。これは、良いフィルターと言えるでしょうか。


> 今日の重要概念（Python）：if
>
> ifは、条件が成り立つ（True）ときだけ、字下げしたブロックを実行する。elseの下は、成り立たないときに動く。


### TODO①：探す語を替える（予想 → 実行 → 比べる）

下のセルは、語`word`を含むなら迷惑と判定し、迷惑と判定した通数と正解率を表示します。2節のお手本と同じ4行で、語を変数`word`にしまったので、替えるのは2行目だけです。

まず、予想を書いてください。探す語を'xyz'（どのメッセージにも出てこない語）にすると、迷惑と判定する通数と正解率は、いくつになるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`'free'`を`'xyz'`に替えて実行してください。


In [ ]:
# TODO: 'free' を 'xyz' に替えて実行する

word = 'xyz'
pred_x = []
for text in df['本文']:
    pred_x.append(rule(text, word))
df['判定_x'] = pred_x
print(df['判定_x'].sum())   # 迷惑と判定した通数
print(round(accuracy_score(df['迷惑'], df['判定_x']), 3))   # 正解率


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。何も迷惑と判定しないルールの正解率を見て、気づいたことを書いてください。


（ここに答えを書く）


'xyz'を含むメッセージは0通なので、全部を「普通」と判定します。それでも、正解率は0.874です。freeのルール（0.894）と、ほとんど変わりません。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| ルールで見分ける（1・2節） | 「freeを含むなら迷惑」のルールで、正解率0.894 |
| 正解率を疑う（3節） | ？（このあとの段で） |
| 2つの割合（4節） | ？（このあとの段で） |
| 確率を出す（5節） | ？（このあとの段で） |
| しきい値（6節） | ？（このあとの段で） |

まだ分からないのは、正解率0.894が、本当に良い数なのか、です。


---


## 3. 正解率は信じてよい？


ここから、段2「正解率を疑う」（3節）です。問いは、正解率0.894を信じてよいか、です。

freeのルールは正解率0.894でしたが、何も迷惑と言わないルールでも0.874でした。差はわずかです。freeのルールは、何を当てて、何を外しているのでしょうか。


### なぜ87%も当たる？

迷惑は8通に1通（0.126）で、普通は0.874です。全部を「普通」と答えると、普通の4,518通がすべて当たります。正解率は、多い側（普通）の当たりで、ほとんど決まってしまいます。

正解率は、どこで当てて、どこで外したかを1つの数にまとめます。迷惑を1通も捕まえないフィルターでも、87%になります。当たりと外れを、種類に分けて数えたいのです。


### 本当と判定で、4つのマス

本当の答え（迷惑・普通）と、判定（迷惑・普通）の組で、メッセージを4つに分けます。

| | 判定：普通 | 判定：迷惑 |
|---|---|---|
| 本当：普通 | 普通を当てた（TN） | 誤検出（FP） |
| 本当：迷惑 | 見逃し（FN） | 迷惑を当てた（TP） |

普通（0）を先に、迷惑（1）を後に並べます。このあと使う`sklearn`の道具と同じ並びです。

4つのマスに通数を数えた表を、混同行列（confusion matrix）と呼びます。見逃しは、迷惑を普通と判定する誤りで、偽陰性（false negative、FN）とも呼びます。誤検出は、普通を迷惑と判定する誤りで、偽陽性（false positive、FP）とも呼びます。迷惑を当てたマスは真陽性（TP）、普通を当てたマスは真陰性（TN）です。


### お手本：混同行列を数える

- 役割：本当の答えと判定から、4つのマスの通数を数える
- 入力：列「迷惑」（本当の答え）と列「判定」
- 出力：2×2の表

`sklearn`の`confusion_matrix(本当, 判定)`は、4つのマスの通数を2×2の表で返す道具です。


In [ ]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(df['迷惑'], df['判定']))


#### 1行ずつ読む

- 1行目：`sklearn`の評価の道具の中から、`confusion_matrix`を呼び出す
- 2行目：本当の答えと判定を渡すと、2×2の表が返る。それを表示する
- 渡す順は、本当が先、判定が後


#### confusion_matrixがしていること

1通ずつ、本当の答え（行）と判定（列）の組を見て、当てはまるマスに1を足します。正解率のときと同じ5通なら、次のようになります。

| 行 | 本当 | 判定 | マス |
|---:|---:|---:|:---:|
| 0 | 0 | 0 | TN |
| 1 | 0 | 0 | TN |
| 2 | 1 | 1 | TP |
| 8 | 1 | 0 | FN |
| 75 | 0 | 1 | FP |

この5通なら、TNが2、TP・FN・FPが1ずつです。これを5,171通で数えたのが、上の4つの数です。

並びは、上の行が本当の普通、下の行が本当の迷惑、左の列が判定の普通、右の列が判定の迷惑です。`sklearn`は0、1の順に並べます。教科書やネットの記事には、迷惑（1）を左上に置く表もあるので、表を読むときは、行と列の名前を先に確かめます。


#### 中身を見る：5通の混同行列


In [ ]:
y = [0, 0, 1, 1, 0]      # 本当（行0・1・2・8・75）
p = [0, 0, 1, 0, 1]      # 判定
print(confusion_matrix(y, p))


上の表と同じ5通。TNが2、FP・FN・TPが1ずつ。上の行が本当の0（普通）、左の列が判定の0（普通）


In [ ]:
# @title 図：混同行列を2×2の表に描く（中身は読み飛ばしてよい）
vals = confusion_matrix(df['迷惑'], df['判定'])
plt.figure(figsize=(5, 4))
plt.imshow([[1, 0.5], [0, 1]], cmap='Pastel1', vmin=0, vmax=8)
names = [['普通を当てた（TN）', '誤検出（FP）'], ['見逃し（FN）', '迷惑を当てた（TP）']]
for i in range(2):
    for j in range(2):
        plt.text(j, i - 0.15, names[i][j], ha='center', fontsize=11)
        plt.text(j, i + 0.15, vals[i][j], ha='center', fontsize=18)
plt.xticks([0, 1], ['判定：普通', '判定：迷惑'])
plt.yticks([0, 1], ['本当：普通', '本当：迷惑'])
plt.title('freeのルール')
plt.show()


迷惑653通のうち、freeのルールで捕まえたのは169通です。484通を見逃しています。普通4,518通のうち、62通を誤って迷惑と判定しました。

正解率は、当たりの2つのマス（169と4,456）を全体で割ったものです。当たりのほとんどは、普通を普通と言った4,456通です。何も迷惑と言わないルールなら、迷惑653通をすべて見逃しますが、それでも正解率は0.874になります。


> 今日の重要概念（統計）：混同行列
>
> 正解率だけで見ない。混同行列の4つのマスを数え、見逃しと誤検出を分けて評価する。


### TODO②：探す語をprizeに（予想 → 実行 → 比べる）

下のセルは、語`word`を含むなら迷惑と判定し、混同行列を表示します。prizeは「賞品」で、普通のメッセージにはめったに出ない語です。

まず、予想を書いてください。探す語をprizeにすると、見逃しと誤検出は、freeと比べてどう変わるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`'free'`を`'prize'`に替えて実行してください。


In [ ]:
# TODO: 'free' を 'prize' に替えて実行する

word = 'prize'
pred_x = []
for text in df['本文']:
    pred_x.append(rule(text, word))
df['判定_x'] = pred_x
print(confusion_matrix(df['迷惑'], df['判定_x']))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。見逃しと誤検出は、それぞれどう変わりましたか。


（ここに答えを書く）


prizeでは、誤検出は0通です。でも、見逃しは576通に増えました。prizeは迷惑にしか出てこないので、迷惑と言えば必ず当たります。でも、prizeを含む迷惑は少ないので、ほとんどを見逃します。freeとprizeのどちらが良いかは、2つの誤りをどう比べるかで決まります。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| ルールで見分ける（1・2節） | 「freeを含むなら迷惑」のルールで、正解率0.894 |
| 正解率を疑う（3節） | 何も迷惑と言わなくても0.874。4つのマスで数える |
| 2つの割合（4節） | ？（このあとの段で） |
| 確率を出す（5節） | ？（このあとの段で） |
| しきい値（6節） | ？（このあとの段で） |

まだ分からないのは、見逃しと誤検出を、どんな数で比べればよいか、です。


---


## 4. 見逃しと誤検出は？


ここから、段3「2つの割合」（4節）です。問いは、見逃しと誤検出を、どんな数で比べればよいか、です。

混同行列で、見逃しと誤検出を分けて数えられました。でも、通数のままでは、迷惑の多いデータと少ないデータで比べにくいです。割合にして、2つの数で評価します。


### 再現率と適合率

再現率（recall）は、本当に迷惑の通のうち、迷惑と判定できた割合です。見逃しが少ないほど、1に近くなります。

$$
\text{再現率} = \frac{TP}{TP + FN}
$$

適合率（precision）は、迷惑と判定した通のうち、本当に迷惑だった割合です。誤検出が少ないほど、1に近くなります。

$$
\text{適合率} = \frac{TP}{TP + FP}
$$


### お手本：2つの割合

- 役割：本当の答えと判定から、再現率と適合率を出す
- 入力：列「迷惑」と列「判定」
- 出力：再現率と適合率（小数第3位まで）

`recall_score(本当, 判定)`は再現率、`precision_score(本当, 判定)`は適合率を返す道具です。


In [ ]:
from sklearn.metrics import recall_score, precision_score
print(round(recall_score(df['迷惑'], df['判定']), 3))      # 再現率
print(round(precision_score(df['迷惑'], df['判定']), 3))   # 適合率


#### 1行ずつ読む

- 1行目：再現率の道具`recall_score`と、適合率の道具`precision_score`を呼び出す
- 2行目：本当の答えと判定を渡し、再現率を出す
- 3行目：同じ2つを渡し、適合率を出す


#### 2つの道具がしていること

どちらも、混同行列の4つのマスのうち、2つだけを使います。

- `recall_score`：下の行（本当に迷惑）だけを見る。TP ÷ (TP + FN) = 169 ÷ (169 + 484)
- `precision_score`：右の列（迷惑と判定）だけを見る。TP ÷ (TP + FP) = 169 ÷ (169 + 62)


#### 中身を見る：5通の2つの割合


In [ ]:
y = [0, 0, 1, 1, 0]      # 本当（行0・1・2・8・75）
p = [0, 0, 1, 0, 1]      # 判定
print(recall_score(y, p), precision_score(y, p))


再現率：本当に迷惑の2通（行2・8）のうち、迷惑と判定したのは1通。適合率：迷惑と判定した2通（行2・75）のうち、本当に迷惑は1通


freeのルールは、再現率0.259で、迷惑の4通に1通しか捕まえていません。適合率は0.732で、迷惑と言った4通のうち3通ほどは本当に迷惑です。正解率0.894からは見えなかった弱さが、再現率に出ました。


### お手本：prizeのルールの2つの割合

- 役割：prizeを含むなら迷惑、のルールで判定し、再現率と適合率を出す
- 入力：表`df`
- 出力：再現率と適合率


In [ ]:
pred_p = []
for text in df['本文']:
    pred_p.append(rule(text, 'prize'))
df['判定_p'] = pred_p
print(round(recall_score(df['迷惑'], df['判定_p']), 3))
print(round(precision_score(df['迷惑'], df['判定_p']), 3))


#### 1行ずつ読む

- 1〜4行目：2節と同じ形。探す語をprizeにして、列「判定_p」を作る
- 5・6行目：再現率と適合率を出す


prizeは適合率1.0ですが、再現率は0.118です。1つの語では、迷惑の一部しか捕まえられません。


### 迷惑には、番号が並ぶ

迷惑の本文には、電話番号や、返信先の短い番号が書いてあります。普通のメッセージに、数字はあまり出てきません。本文の中の数字を数えたら、手がかりにならないでしょうか。

文字列も、`for ch in 文字列:`で1字ずつ取り出せます。`isdigit()`は、その字が数字ならTrueを返します。


In [ ]:
for ch in 'a1b2':
    print(ch, ch.isdigit())


### お手本：数字を数える関数

- 役割：本文を1字ずつ見て、数字の字数を数える
- 入力：本文`text`
- 出力：数字の字数


In [ ]:
def n_digits(text):
    n = 0
    for ch in text:          # 本文を1字ずつ
        if ch.isdigit():     # 数字なら
            n = n + 1        # 1を足す
    return n

print(n_digits('call 08707509020 now'))


#### 1行ずつ読む

- 1行目：関数`n_digits`を作る。入力は本文`text`
- 2行目：数えた数をためる変数`n`を、0から始める
- 3行目：本文を1字ずつ取り出して`ch`に入れる
- 4行目：`ch`が数字のときだけ、5行目へ進む
- 5行目：`n`に1を足して、`n`にしまい直す
- 6行目：全部の字を見終わったら`n`を返す。字下げが`for`の外なので、最後に1回だけ動く
- 8行目：電話番号の入った本文で試す


#### 中身を見る：1字ずつのnの変化


In [ ]:
n = 0
for ch in 'call 0870':
    if ch.isdigit():
        n = n + 1
    print(ch, n)


数字の字のところでだけ、`n`が1つ増える


### お手本：数字の数の列

- 役割：全部の通で数字を数えて列「数字」にし、迷惑と普通で平均を比べる
- 入力：表`df`の本文
- 出力：迷惑（1）と普通（0）の、数字の字数の平均


In [ ]:
nd = []
for text in df['本文']:
    nd.append(n_digits(text))
df['数字'] = nd
print(df.groupby('迷惑')['数字'].mean().round(1))


#### 1行ずつ読む

- 1〜4行目：2節の判定と同じ形。1通ずつ`n_digits`で数え、リストにためて列「数字」にする
- 5行目：`groupby('迷惑')`で迷惑（1）と普通（0）に分け、列「数字」の平均を出す
- `round(1)`で、小数第1位までに丸める


迷惑は平均15.5字、普通は平均0.3字です。


In [ ]:
# @title 図：数字の字数を、迷惑と普通で描き比べる（中身は読み飛ばしてよい）
plt.hist(df[df['迷惑'] == 0]['数字'], bins=range(0, 48, 2), alpha=0.6, label='普通')
plt.hist(df[df['迷惑'] == 1]['数字'], bins=range(0, 48, 2), alpha=0.6, label='迷惑')
plt.yscale('log')
plt.xlabel('本文の中の数字の字数')
plt.ylabel('通数（対数の目盛り）')
plt.legend()
plt.show()


普通はほとんどが0字で、迷惑は数字が何字も並ぶ通が多くあります。数字の数で、ほぼ分かれます。


### お手本：数字のルール

- 役割：数字が5字以上なら迷惑と判定し、混同行列と2つの割合を出す
- 入力：表`df`の列「数字」
- 出力：混同行列と、再現率・適合率


In [ ]:
pred_d = []
for n in df['数字']:
    if n >= 5:
        pred_d.append(1)     # 5字以上なら迷惑
    else:
        pred_d.append(0)
df['判定_数字'] = pred_d
print(confusion_matrix(df['迷惑'], df['判定_数字']))
print(round(recall_score(df['迷惑'], df['判定_数字']), 3))
print(round(precision_score(df['迷惑'], df['判定_数字']), 3))


#### 1行ずつ読む

- 2行目：列「数字」から、1通ずつ字数を取り出して`n`に入れる
- 3〜6行目：`n`が5以上なら1、そうでなければ0を、`pred_d`の後ろに足す
- `rule`と違い、ifを関数に入れず、くり返しの中に直接書いた
- 7行目：リストを列「判定_数字」にする
- 8行目：混同行列を表示する
- 9・10行目：再現率と適合率を表示する


見逃しは、freeのルールの484通から96通に減りました。誤検出は39通です。再現率0.853、適合率0.935で、freeのルールより、どちらも良くなりました。


> 重要：2つの誤りの重さは違う
>
> 見逃しは、迷惑が届いてしまう。誤検出は、大事な連絡が届かない。重さが違うので、2つを分けて見る。


### TODO③：数字のしきいを替える（予想 → 実行 → 比べる）

下のセルは、数字が`k`字以上なら迷惑と判定し、混同行列と2つの割合を表示します。

まず、予想を書いてください。数字が1字でもあれば迷惑、にすると、再現率と適合率はどうなるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`5`を`1`に替えて実行してください。


In [ ]:
# TODO: 5 を 1 に替えて実行する

k = 1
pred_x = []
for n in df['数字']:
    if n >= k:
        pred_x.append(1)
    else:
        pred_x.append(0)
df['判定_x'] = pred_x
print(confusion_matrix(df['迷惑'], df['判定_x']))
print(round(recall_score(df['迷惑'], df['判定_x']), 3))
print(round(precision_score(df['迷惑'], df['判定_x']), 3))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。再現率と適合率は、それぞれどう変わりましたか。


（ここに答えを書く）


再現率は0.94に上がりました。でも、誤検出が715通に増え、適合率は0.462に下がりました。しきいを下げると、迷惑を広く捕まえますが、普通も巻き込みます。しきいを上げると、確かなものだけを捕まえ、見逃しが増えます。再現率と適合率の両方を、同時に1にするのは難しいのです。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| ルールで見分ける（1・2節） | 「freeを含むなら迷惑」のルールで、正解率0.894 |
| 正解率を疑う（3節） | 何も迷惑と言わなくても0.874。4つのマスで数える |
| 2つの割合（4節） | 数字が5字以上なら迷惑：見逃し96・誤検出39 |
| 確率を出す（5節） | ？（このあとの段で） |
| しきい値（6節） | ？（このあとの段で） |

まだ分からないのは、数字と語を、まとめて使えないか、です。


---


## 5. 手がかりをまとめると？


ここから、段4「確率を出す」（5節）です。問いは、数字と語の手がかりを、まとめて1つの確率にできないか、です。

数字のルールは、見逃し96通・誤検出39通でした。迷惑の決まり文句（free、claim、prize…）も、手がかりになりました。手がかりを、迷惑である確率にまとめ、確率で判定します。


### お手本：1つの語の列を作る関数

- 役割：1通ずつ`rule`で判定し、全部の通の1/0のリストを返す
- 入力：探す語`word`
- 出力：全部の通の1/0のリスト

2節では、語ごとに、1通ずつ`rule`で判定してリストにためました。語を15個にするなら、同じ4行を15回書くことになります。そこで、1回分を関数にまとめ、語をくり返しの外から渡します。中身は2節の4行と同じです。


In [ ]:
def rule_col(word):
    pred = []
    for text in df['本文']:
        pred.append(rule(text, word))
    return pred


#### 1行ずつ読む

- 1行目：関数`rule_col`を作る。入力は探す語`word`
- 2〜4行目：2節の4行と同じ。`'free'`のところが`word`になった
- 5行目：ためたリスト`pred`を返す


### お手本：15語の列

- 役割：迷惑によく出てくる15語のそれぞれで、含めば1・なければ0の列を作り、数字の数と合わせて手がかりの列の名前`cols`にする
- 入力：語のリスト`words`
- 出力：手がかりの本数


<details><summary>書き方メモ（読み飛ばしてよい）</summary>

15語は、迷惑メッセージによく出てくる語を、こちらで選びました。本当は、語を選ぶのも、訓練の組だけを見て行うのが公平です。全部の通を見て選ぶと、テストの通の答えを少しのぞいたことになります（第2回のリーク）。

</details>


In [ ]:
words = ['free', 'txt', 'claim', 'prize', 'call', 'win', '£',
         'www', 'urgent', 'mobile', 'reply', 'stop', 'cash',
         'service', 'new']
for w in words:
    df[w] = rule_col(w)          # 語を含めば1、なければ0
cols = ['数字'] + words
print(len(words), '語と数字で', len(cols), '本の手がかり')


#### 1行ずつ読む

- 1〜3行目：15個の語を並べたリストを`words`にしまう。長いので3行に折り返した
- 4行目：`words`から1語ずつ取り出して`w`に入れる
- 5行目：その語を含めば1・なければ0のリストを作り、語の名前の列にする
- 6行目：「数字」と15語をつないだリスト（`+`でつなぐ）。7行目：本数を表示


#### 中身を見る：語の列


In [ ]:
df[['free', 'txt', 'claim', 'call', '数字']].head(3)


語の列は、その語を含めば1。3通めは、freeを含むので1


### 0と1を、確率で当てる

$x_{i1}, \dots, x_{i,16}$は、$i$番目の通の16本の手がかりです。まず、手がかりの1次式の値$z_i$を作り、それをシグモイドという関数で0〜1に収めます。$\hat{p}_i$が、迷惑である確率です。この式の作り方を、ロジスティック回帰（logistic regression）と呼びます。

$$
z_i = b_0 + b_1 x_{i1} + \cdots + b_{16} x_{i,16}, \qquad
\hat{p}_i = \frac{1}{1 + e^{-z_i}}
$$

シグモイドは、$z$が大きいほど1に、小さいほど0に近づき、$z = 0$でちょうど0.5になります。どんな$z$でも、0〜1の間に収まります。

式を作った通で確率を出すと、甘くなります（第2回の過学習）。そこで、第4回の交差検証と同じように、5つの組に分け、1つの組を残して式を作り、残した組の通の確率を出します。第4回は、この手順を自分で書きました。今日は、道具`cross_val_predict`で行います。


### お手本：交差検証で確率を出す

- 役割：交差検証で、全部の通の「迷惑である確率」を出す
- 入力：式の作り方`LogisticRegression()`、手がかりの列`df[cols]`、答えの列`df['迷惑']`
- 出力：全部の通の確率（列「確率」）

`cross_val_predict(式の作り方, 手がかり, 答え, cv=5)`は、交差検証で全部の通の予測を出す道具です。`cv=5`は、5つの組に分けるという意味です。`method='predict_proba'`を付けると、判定（0か1）ではなく、確率を返します。


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
p = cross_val_predict(LogisticRegression(), df[cols], df['迷惑'],
                      cv=5, method='predict_proba')
df['確率'] = p[:, 1]
print(round(df['確率'].min(), 3), round(df['確率'].max(), 3))


#### 1行ずつ読む

- 1・2行目：ロジスティック回帰の道具と、交差検証の道具を呼び出す
- 3・4行目：式の作り方、16本の手がかりの列、答えの列を渡す。`cv=5`は5つの組。`method='predict_proba'`で、全部の通の確率が`p`に返る
- 1行に長いので、3行目の途中で折り返した（かっこの中なら折り返せる）
- 5行目：`p`の右の列（迷惑の確率）を取り出し、列「確率」にしまう
- 6行目：確率のいちばん小さい値と、いちばん大きい値を表示する


確率は0.005〜1.0です。0〜1をはみ出しません。


#### predict_probaがしていること

`p`には、1通に2つの確率が入っています。左の列（0番）が普通である確率、右の列（1番）が迷惑である確率で、2つを足すと1です。`p[:, 1]`は「すべての行の、1番の列」という意味で、迷惑の確率だけを取り出します。下のセルで、行0・1・2・8・75の5通を見てみます。


In [ ]:
print(p[[0, 1, 2, 8, 75]].round(3))


#### cross_val_predictがしていること

表を上から順に5つの組に分けます。1回目は、2〜5番目の組で式を作り、1番目の組の通の確率を出します。2回目は、2番目の組を残します。これを5回くり返すと、どの通も、自分を見ていない式から確率をもらいます。

| 回 | 1組 | 2組 | 3組 | 4組 | 5組 |
|---|---|---|---|---|---|
| 1回目 | 確率を出す | 式を作る | 式を作る | 式を作る | 式を作る |
| 2回目 | 式を作る | 確率を出す | 式を作る | 式を作る | 式を作る |
| … | | | | | |
| 5回目 | 式を作る | 式を作る | 式を作る | 式を作る | 確率を出す |


#### 中身を見る：確率の列


In [ ]:
df[['迷惑', '確率']].head(3)


1通に1つずつ、確率が付いた。迷惑（1）の行ほど確率が高い


### お手本：確率を描く

- 役割：迷惑と普通で、確率の分かれ方を描き比べる
- 入力：列「確率」と列「迷惑」
- 出力：2色の柱の図

`plt.yscale('log')`は、縦軸を10倍ごとの目盛りにします。少ない迷惑も見えるようにするためです。


In [ ]:
ham = df[df['迷惑'] == 0]['確率']
spam = df[df['迷惑'] == 1]['確率']
plt.hist(ham, bins=40, alpha=0.6, label='普通')
plt.hist(spam, bins=40, alpha=0.6, label='迷惑')
plt.yscale('log')
plt.xlabel('迷惑である確率')
plt.ylabel('通数（対数の目盛り）')
plt.legend()
plt.show()


#### 1行ずつ読む

- 1行目：普通の行だけを取り出し、確率の列を`ham`にしまう
- 2行目：迷惑の行の確率を`spam`にしまう
- 3・4行目：確率を40の区間に分けて柱を描く（`hist`）
- `alpha=0.6`で半透明にし、重なりを見せる。`label`は凡例に出す名前
- 5行目：縦軸を10倍ごとの目盛りにする
- 6〜8行目：軸の名前と凡例を付ける。9行目：図を表示する


普通は0のまわり、迷惑は1のまわりに多くあります。迷惑ほど、確率が高くなりました。


### お手本：確率を切って判定する

- 役割：確率が0.5以上なら迷惑と判定し、混同行列と2つの割合を出す
- 入力：列「確率」
- 出力：混同行列と、再現率・適合率

`df['確率'] >= 0.5`は、0.5以上ならTrue、違えばFalseの列です。`astype(int)`で1と0に直します。0節で列「迷惑」を作ったのと同じ書き方です。


In [ ]:
df['判定_式'] = (df['確率'] >= 0.5).astype(int)
print(confusion_matrix(df['迷惑'], df['判定_式']))
print(round(recall_score(df['迷惑'], df['判定_式']), 3))
print(round(precision_score(df['迷惑'], df['判定_式']), 3))


#### 1行ずつ読む

- 1行目：`df['確率'] >= 0.5`は、1通ずつ0.5と比べ、以上ならTrueの列。`astype(int)`で1と0に直し、列「判定_式」にしまう
- 2行目：混同行列を表示する
- 3・4行目：再現率と適合率を表示する


#### 中身を見る：確率と判定


In [ ]:
df[['確率', '判定_式']].head(3)


0.5以上の行だけが1になった


In [ ]:
# @title 図：混同行列を2×2の表に描く（中身は読み飛ばしてよい）
vals = confusion_matrix(df['迷惑'], df['判定_式'])
plt.figure(figsize=(5, 4))
plt.imshow([[1, 0.5], [0, 1]], cmap='Pastel1', vmin=0, vmax=8)
names = [['普通を当てた（TN）', '誤検出（FP）'], ['見逃し（FN）', '迷惑を当てた（TP）']]
for i in range(2):
    for j in range(2):
        plt.text(j, i - 0.15, names[i][j], ha='center', fontsize=11)
        plt.text(j, i + 0.15, vals[i][j], ha='center', fontsize=18)
plt.xticks([0, 1], ['判定：普通', '判定：迷惑'])
plt.yticks([0, 1], ['本当：普通', '本当：迷惑'])
plt.title('確率を0.5で切る')
plt.show()


迷惑である確率が半分以上なら、迷惑と判定しました。見逃しは83通、誤検出は23通です。数字のルール（見逃し96・誤検出39）より、どちらも少なくなりました。16本の手がかりを、1つの確率にまとめた効果です。


> 重要：確率を切って判定する
>
> ロジスティック回帰は、手がかりの1次式をシグモイドで0〜1に収め、迷惑である確率を出す。確率をしきい値で切ると、判定になる。


### TODO④：手がかりから数字を外す（予想 → 実行 → 比べる）

下のセルは、手がかりの列`cols_x`で交差検証の確率を出し、0.5で切った混同行列を表示します。

まず、予想を書いてください。手がかりを15語だけにする（数字を外す）と、見逃しは83通から増えるでしょうか、減るでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`['数字'] + words`を`words`に替えて実行してください。


In [ ]:
# TODO: ['数字'] + words を words に替えて実行する

cols_x = words
p = cross_val_predict(LogisticRegression(), df[cols_x], df['迷惑'],
                      cv=5, method='predict_proba')
df['判定_x'] = (p[:, 1] >= 0.5).astype(int)
print(confusion_matrix(df['迷惑'], df['判定_x']))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。数字を外すと、見逃しと誤検出はどう変わりましたか。


（ここに答えを書く）


15語だけでは、見逃しが175通に増えました。16本の手がかりの中で、数字の数がいちばん効く手がかりです。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| ルールで見分ける（1・2節） | 「freeを含むなら迷惑」のルールで、正解率0.894 |
| 正解率を疑う（3節） | 何も迷惑と言わなくても0.874。4つのマスで数える |
| 2つの割合（4節） | 数字が5字以上なら迷惑：見逃し96・誤検出39 |
| 確率を出す（5節） | 数字と15語で確率。0.5で切ると見逃し83・誤検出23 |
| しきい値（6節） | ？（このあとの段で） |

まだ分からないのは、0.5で切ったのは正しかったのか、です。


---


## 6. しきい値を動かすと？


ここから、段5「しきい値」（6節）です。問いは、確率を切る場所を動かすと、見逃しと誤検出はどう変わるか、です。

0.5で切ると、見逃しは83通、誤検出は23通でした。見逃しをもっと減らしたいなら、0.5より下で切ればよいはずです。確率を切る場所を、しきい値（threshold）と呼びます。しきい値を下げると、迷惑をより多く捕まえます。


### お手本：しきい値ごとに数える

- 役割：しきい値を0.1〜0.9に動かし、見逃しと誤検出をリストにためて、表にする
- 入力：列「確率」
- 出力：しきい値ごとの見逃しと誤検出の表

`m = confusion_matrix(…)`の2×2の表から、1マスを`m[行, 列]`で取り出します（5節の`p[:, 1]`と同じ形です）。番号は0から数えます。見逃しは下の行・左の列なので`m[1, 0]`、誤検出は上の行・右の列なので`m[0, 1]`です。


In [ ]:
ts = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
fns = []
fps = []
for t in ts:
    pred = (df['確率'] >= t).astype(int)
    m = confusion_matrix(df['迷惑'], pred)
    fns.append(m[1, 0])          # 見逃し
    fps.append(m[0, 1])          # 誤検出
print(pd.DataFrame({'しきい値': ts, '見逃し': fns, '誤検出': fps}))


#### 1行ずつ読む

- 1行目：しきい値の候補9つのリスト`ts`
- 2・3行目：見逃しと誤検出をためる、空のリストを2つ作る
- 4行目：`t`に0.1、0.2、…を順に入れ、字下げした4行をくり返す
- 5行目：確率が`t`以上なら1の判定を作り、`pred`にしまう
- 6行目：混同行列を作り、`m`にしまう
- 7行目：`m[1, 0]`（見逃し）を、`fns`の後ろに足す
- 8行目：`m[0, 1]`（誤検出）を、`fps`の後ろに足す
- 9行目：3つのリストを列にした表を作り、表示する


#### 中身を見る：0.5のときのm


In [ ]:
m = confusion_matrix(df['迷惑'], (df['確率'] >= 0.5).astype(int))
print(m)
print(m[1, 0], m[0, 1])


`m[1, 0]`は下の行・左の列で83（見逃し）、`m[0, 1]`は上の行・右の列で23（誤検出）


### お手本：見逃しと誤検出を描く

- 役割：しきい値を横軸に、見逃しと誤検出の2本の線を描く
- 入力：`ts`・`fns`・`fps`
- 出力：2本の折れ線


In [ ]:
plt.plot(ts, fns, 'o-', label='見逃し（FN）')
plt.plot(ts, fps, 's-', label='誤検出（FP）')
plt.xlabel('しきい値')
plt.ylabel('通数')
plt.legend()
plt.show()


#### 1行ずつ読む

- 1行目：横にしきい値、縦に見逃しの線を描く。`'o-'`は丸い点を線でつなぐ
- 2行目：誤検出の線。`'s-'`は四角い点
- 3〜6行目：軸の名前と凡例を付けて、表示する


しきい値0.1では、見逃し33・誤検出85です。0.9では、見逃し143・誤検出9です。2本の線は、逆向きに動きます。


> 重要：しきい値で入れ替わる
>
> しきい値を下げると見逃しが減って誤検出が増え、上げるとその逆になる。どこで切るかで、同じ式の評価が変わる。


### 全部のしきい値を1枚に：ROC曲線

誤検出率は、普通の通のうち、迷惑と判定してしまった割合です。

$$
\text{誤検出率} = \frac{FP}{FP + TN}
$$

横に誤検出率、縦に再現率をとり、しきい値を細かく動かして点を打つと、1本の線になります。この線を、ROC曲線と呼びます。左上に近いほど、良い確率です。

`roc_curve(本当, 確率)`は、しきい値を少しずつ下げ、そのたびに誤検出率`fpr`と再現率`tpr`を出して並べる道具です。上の表の9つのしきい値の点も、この線の上に乗ります。


In [ ]:
from sklearn.metrics import roc_curve
fpr, tpr, th = roc_curve(df['迷惑'], df['確率'])
plt.plot(fpr, tpr)
plt.xlabel('誤検出率')
plt.ylabel('再現率')
plt.show()


#### 1行ずつ読む

- 1行目：ROC曲線の道具`roc_curve`を呼び出す
- 2行目：本当の答えと確率を渡すと、3つが返る。誤検出率の並び`fpr`、再現率の並び`tpr`、しきい値の並び`th`
- 3行目：誤検出率を横、再現率を縦にして、点を線で結ぶ
- 4〜6行目：軸の名前を付けて、図を表示する


#### 中身を見る：3つの並びの長さ


In [ ]:
print(len(fpr), len(tpr), len(th))


しきい値を276通り試し、そのたびに誤検出率と再現率を出した


下の図は、横軸を0.2までに広げ、これまでのルール（free・prize・数字5字以上）の点も打ったものです。


In [ ]:
# @title 図：ROC曲線と、ルールの点（中身は読み飛ばしてよい）
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label='式の確率（しきい値を動かす）')
for name, col in [('free', '判定'), ('prize', '判定_p'), ('数字5字以上', '判定_数字')]:
    m = confusion_matrix(df['迷惑'], df[col])
    x = m[0, 1] / (m[0, 1] + m[0, 0])
    y = m[1, 1] / (m[1, 1] + m[1, 0])
    plt.plot(x, y, 'o', color='red')
    plt.annotate(name, (x, y), xytext=(8, -4), textcoords='offset points', color='red')
plt.xlim(-0.005, 0.2)
plt.xlabel('誤検出率（普通のうち、迷惑と判定した割合）')
plt.ylabel('再現率')
plt.legend(loc='lower right')
plt.show()


赤い点は、これまでのルールです。どれも、式の線より下（右）にあります。同じ誤検出率なら、式のほうが多くの迷惑を捕まえます。


### お手本：AUCを出す

- 役割：ROC曲線の下の面積（AUC）を出す
- 入力：列「迷惑」と列「確率」
- 出力：AUC

ROC曲線の下の面積を、AUC（area under the curve）と呼びます。1に近いほど、迷惑と普通を確率でよく分けています。でたらめな確率なら0.5です。しきい値を決めなくても出せる、確率の付き方そのものの良さです。`roc_auc_score(本当, 確率)`に、しきい値は渡しません。


In [ ]:
from sklearn.metrics import roc_auc_score
print(round(roc_auc_score(df['迷惑'], df['確率']), 3))


#### 1行ずつ読む

- 1行目：AUCの道具`roc_auc_score`を呼び出す
- 2行目：本当の答えと確率を渡すと、AUCが返る。しきい値は渡さない


#### roc_auc_scoreがしていること

迷惑1通と普通1通を組にして、迷惑のほうが確率の高い組を数えます。たとえば、迷惑の通の確率が0.9、普通の通の確率が0.2なら、その組は迷惑のほうが高い（○）です。迷惑の通が0.1、普通の通が0.2なら×です。迷惑653通と普通4,518通のすべての組でこれを数えた割合が、AUCと同じになります。この式では、0.981の割合の組で、迷惑のほうが確率が高くなっています。


### お手本：LDAでも確率を出す

- 役割：線形判別分析（linear discriminant analysis）で確率を出し、0.5で切って、正解率と混同行列をロジスティック回帰と比べる
- 入力：式の作り方`LinearDiscriminantAnalysis()`
- 出力：正解率と、混同行列

線形判別分析は、英語の頭文字からLDAとも呼びます。式の作り方を、`LogisticRegression()`から`LinearDiscriminantAnalysis()`に替えるだけです。


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
p = cross_val_predict(LinearDiscriminantAnalysis(), df[cols],
                      df['迷惑'], cv=5, method='predict_proba')
df['判定_lda'] = (p[:, 1] >= 0.5).astype(int)
print(round(accuracy_score(df['迷惑'], df['判定_lda']), 3))
print(confusion_matrix(df['迷惑'], df['判定_lda']))


#### 1行ずつ読む

- 1行目：LDAの道具`LinearDiscriminantAnalysis`を呼び出す
- 2・3行目：5節の確率のお手本と同じ形。式の作り方だけを替えた
- 4行目：0.5で切って、列「判定_lda」にしまう
- 5・6行目：正解率と混同行列を表示する


正解率は0.973で、ロジスティック回帰（0.98）とほとんど同じです。下のセルで、LDAのAUCも出してみます。


In [ ]:
print(round(roc_auc_score(df['迷惑'], p[:, 1]), 3))


AUCも、ロジスティック回帰と同じ0.981です。1つの数で比べると、2つの式にほとんど差はありません。

ところが、混同行列を並べると、見逃しはロジスティック回帰が83通、LDAが132通です。誤検出は、23通と8通です。LDAは見逃しが多く、誤検出が少ないのです。正解率もAUCも、4つのマスを1つの数にまとめた数で、どちらの誤りで外したかが消えます。

どちらの式が良いかは、誤りの重さで決まります。迷惑フィルターでは、大事な連絡を迷惑フォルダに入れてしまう誤検出のほうが、致命的かもしれません。そうなら、LDAを選ぶこともあります。だから、式どうしを比べるときも、混同行列を並べます（3節と同じです）。

見逃しが増えるのは、LDAの確率が0か1の近くに偏るからです。下のセルで、確率が0か1の近く（0.01未満か0.99より上）にある通の割合を比べます。


In [ ]:
print(round(((df['確率'] < 0.01) | (df['確率'] > 0.99)).mean(), 3))
print(round(((p[:, 1] < 0.01) | (p[:, 1] > 0.99)).mean(), 3))


LDAは0.978、ロジスティック回帰は0.777の割合の通が、0か1の近くにあります。LDAが悪いのではなく、どこで切るのがよいかが、式ごとに違うのです。違いが出たのは、2つのモデルの仮定（データの出方の考え方）が違うからです。このデータではこうなりましたが、いつもLDAが見逃しを多くする、とは言えません。


### しきい値を下げても、残る見逃し

しきい値を0.1まで下げても、見逃しは33通残りました。迷惑なのに、確率がとても小さい通があるのです。どんなメッセージかを読んでみます。


### お手本：見逃した迷惑を読む

- 役割：迷惑なのに、確率が0.05より小さい通を取り出し、本文を読む
- 入力：列「迷惑」と列「確率」
- 出力：通数と、本文の先頭5通

まず迷惑の通だけを`spam_df`に取り出し、その中から確率が0.05より小さい通を取り出します。


In [ ]:
spam_df = df[df['迷惑'] == 1]
low = spam_df[spam_df['確率'] < 0.05]
print(len(low))
low['本文'].head()


#### 1行ずつ読む

- 1行目：迷惑の行だけを取り出し、`spam_df`にしまう
- 2行目：その中から、確率が0.05より小さい行を取り出し、`low`にしまう
- 3行目：`low`の通数を表示する
- 4行目：本文の先頭5通を表示する


迷惑なのに、確率が0.05より小さい通が24通あります。冗談や誘い、お知らせの形で、数字も15語も出てきません。16本の手がかりでは、確率が上がりません。しきい値では救えず、捕まえるには別の手がかりが要ります。


### TODO⑤：しきい値を替える（予想 → 実行 → 比べる）

下のセルは、しきい値`t`で判定し、混同行列と2つの割合を表示します。

まず、予想を書いてください。0.3で切ると、見逃し（0.5では83）と誤検出（0.5では23）は、それぞれどう変わるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`0.5`を`0.3`に替えて実行してください。


In [ ]:
# TODO: 0.5 を 0.3 に替えて実行する

t = 0.3
df['判定_x'] = (df['確率'] >= t).astype(int)
print(confusion_matrix(df['迷惑'], df['判定_x']))
print(round(recall_score(df['迷惑'], df['判定_x']), 3))
print(round(precision_score(df['迷惑'], df['判定_x']), 3))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。0.3で切ると、見逃しと誤検出はどう変わりましたか。


（ここに答えを書く）


0.3で切ると、見逃しは66通に減り、誤検出は34通に増えました。再現率が上がり、適合率が下がります。式の強みは、しきい値で、見逃しと誤検出の釣り合いを選べることです。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| ルールで見分ける（1・2節） | 「freeを含むなら迷惑」のルールで、正解率0.894 |
| 正解率を疑う（3節） | 何も迷惑と言わなくても0.874。4つのマスで数える |
| 2つの割合（4節） | 数字が5字以上なら迷惑：見逃し96・誤検出39 |
| 確率を出す（5節） | 数字と15語で確率。0.5で切ると見逃し83・誤検出23 |
| しきい値（6節） | 0.3で切ると見逃し66・誤検出34。AUCは0.981 |

5つの段がそろい、今日の問いに答えられます。


### 答え

本文の数字と15語で迷惑である確率を出し、0.5で切ると、迷惑653通のうち570通を捕まえます。そのとき、普通4,518通のうち23通を、誤って迷惑と判定します。しきい値を動かせば、この釣り合いを選べます。

フィルターの評価は、正解率ではなく、混同行列の見逃しと誤検出で行います。何も見分けないルールでも、正解率は0.874になるからです。


まだ言えないこと

- どこで切るべきか：見逃しと誤検出の重さから、しきい値をどう決めるか
- 手がかりの外：数字も決まり文句もない迷惑を、どう捕まえるか
- 確率は信じてよいか：確率0.3と言われた通は、本当に3割が迷惑か


---


## 目標に答えられたか


- 今日の目標は「本文から、迷惑メッセージをどう見分け、その見分け方をどう評価するか」でした。混同行列の数で答えられますか？
- 2節で、`rule`の中の`if`と`else`は、それぞれどんなときに動きましたか？
- TODO①で、何も迷惑と言わないルールの正解率が高かったのはなぜですか？
- 3節で、`confusion_matrix`の表の、上の行と左の列は、それぞれ何を表しますか？
- 4節で、再現率と適合率は、それぞれ何の少なさを表しますか？
- 5節で、ロジスティック回帰の確率が0〜1をはみ出さないのはなぜですか？
- 6節で、しきい値を下げると、見逃しと誤検出はどう変わりましたか？AUCは何を測りますか？


---


## 今日の要点


- 今日、持ち帰る1文：迷惑メッセージが少ないと、何も見分けなくても正解率は高くなる。混同行列の4つのマスを数え、見逃しと誤検出を分けて評価する。ロジスティック回帰で迷惑である確率を出すと、どこで切るか（しきい値）で、見逃しと誤検出は入れ替わる
- 重要概念（統計）：混同行列。正解率だけで見ない。4つのマスを数え、見逃しと誤検出を分けて評価する
- 重要概念（Python）：if。条件が成り立つときだけ、字下げしたブロックを実行する。elseの下は、成り立たないときに動く
- 再現率は見逃しの少なさ、適合率は誤検出の少なさ。2つは引っ張り合う
- ロジスティック回帰で確率を出し、しきい値で切る。ROC曲線とAUCで、全部のしきい値を見る


### 分類の広がり

2択を当てる場面は、あちこちにあります。病気の検査は陽性か陰性かを当て、病気の人は少数です。クレジットカードの不正な使い方を見つけるときも、不正はごく一部です。工場の検査で不良品を見つけるときも、不良品はめったに出ません。

どこでも、混同行列で話します。病気の検査では、再現率を感度と呼び、見逃しを減らしたい場面が多くあります。迷惑フィルターでは、大事な連絡を失う誤検出を減らしたい。分野で呼び名は違っても、4つのマスと2つの誤りは同じです。

見つけたいものが少ないほど、正解率は役に立たなくなります。不正がごく一部なら、何もしなくても正解率はほぼ1になります。まず混同行列を作る。今日の流れは、そのまま別のデータに持っていけます。


---


## 次回へ


今日は、しきい値を動かすと、見逃しと誤検出が入れ替わるのを見ました。では、見逃しと誤検出の重さから、しきい値をどう決めればよいでしょうか。


---


## 課題（提出）


提出するもの：絵2枚（画像ファイル）。今日の重要概念の、統計とPythonを1枚ずつです。提出フォームから出してください。期限はありません。

コードは書きません。上手な絵である必要はありません。紙に手で描いて写真を撮っても、描画アプリで描いてもかまいません。


### 今日の重要概念（2つ）

- 統計：混同行列。正解率だけで見ない。混同行列の4つのマスを数え、見逃しと誤検出を分けて評価する（3節）
- Python：if。条件が成り立つ（True）ときだけ、字下げしたブロックを実行する。elseの下は、成り立たないときに動く（2・3節）


### 今日の2枚（絵にする）

2つの重要概念を、1枚ずつ絵にしてください。その考えが1枚で伝わる絵です。数式やコードは描かなくてかまいません。半年後の自分が見て、何の話だったか思い出せる絵を目指します。


<details><summary>詰まったら</summary>

混同行列なら、たとえば「火災報知器の、4つの場合」を描けます。
火事で鳴る、火事なのに鳴らない（見逃し）、料理の煙で鳴る（誤検出）、何もなくて静か、の4つです。3節の2×2の表が手本になります。

</details>


<details><summary>詰まったら</summary>

ifなら、たとえば「駅の自動改札」を描けます。
切符があれば扉が開き、なければ閉まります。2節の「ifの分かれ道」の図のように、1人ずつ、条件で通り道が分かれる様子です。

</details>


---


## 発展（任意）


### すべての単語を列にする

今日は、語を15個だけ選びました。本文に出てくる単語を、すべて列にしたらどうなるでしょうか。`CountVectorizer`は、単語ごとに、含むなら1・含まないなら0の列を作る道具です。

`make_pipeline`は、列を作る道具と式を作る道具をつなぎ、1つの式の作り方にする道具です。これを`cross_val_predict`に渡すと、5回のそれぞれで、式を作る組の本文だけから単語を数えます。


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import make_pipeline

vec = CountVectorizer(binary=True).fit(df['本文'])
print(len(vec.vocabulary_), '種類の単語')


#### 1行ずつ読む

- 1・2行目：単語の列を作る道具と、道具をつなぐ道具を呼び出す
- 4行目：全部の本文から、単語の種類を数える（`binary=True`は、含むなら1・含まないなら0）
- 5行目：単語の種類の数を表示する


全部の通で数えると、単語は8,713種類あります。列が、全部の通（5,171通）より多くなります。それでも式が作れるのは、`LogisticRegression`に、第5回のリッジと同じペナルティが、はじめから入っているからです。


In [ ]:
pipe = make_pipeline(CountVectorizer(binary=True),
                     LogisticRegression())
p = cross_val_predict(pipe, df['本文'], df['迷惑'], cv=5,
                      method='predict_proba')
print(round(roc_auc_score(df['迷惑'], p[:, 1]), 3))
print(confusion_matrix(df['迷惑'], (p[:, 1] >= 0.5).astype(int)))


#### 1行ずつ読む

- 1・2行目：単語の列を作る道具とロジスティック回帰をつなぎ、`pipe`にしまう
- 3・4行目：`pipe`を式の作り方として渡し、交差検証で確率を出す。手がかりは本文そのもの
- 5行目：AUCを表示する。6行目：0.5で切った判定の混同行列を表示する


AUCは0.988で、今日の16本の手がかり（0.981）より高くなりました。0.5で切ると、見逃し95・誤検出5です。16本の式（83・23）と比べると、見逃しは少し多く、誤検出はずっと少なくなります。

試すなら、`LogisticRegression()`を`LogisticRegression(C=0.1)`に替えてみましょう。`C`はペナルティの弱さで、小さいほどペナルティが強くなります（第5回のαと逆の向き）。
